# 03. Первичный пространственный анализ

**ЛР 2, шаг 8.** Цель: получить первые ответы на вопросы заказчика и проверить, что выбранные данные и методы работают. Это ещё не продукт, а проверка гипотез, на основе которой уточняются бэклог и план.

Пример AquaMap отвечает на три вопроса:
1. Сколько точек питьевой воды в каждом районе и какова их плотность?
2. Какая доля жилых зданий находится в пределах 500 м от ближайшей точки?
3. Какие парки остаются без питьевой воды?

Замените вопросы и операции своими. Типовые операции: проекция в метры, буфер, пространственное объединение (spatial join), ближайший сосед, пересечение (overlay), агрегирование по районам, картограмма (choropleth).\n\n[![Открыть в Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/OWNER/REPO/blob/main/notebooks/03_spatial_analysis.ipynb)\n\nЧтобы кнопка открывала ваш ноутбук, замените в ссылке `OWNER/REPO` на владельца и имя своего репозитория (например, `ivan-p/pm-lab-00-aquamap`). Другие способы открыть ноутбук описаны в приложении Б.

## Настройка среды

Ноутбук рассчитан на **Google Colab** (как получить бесплатный доступ: приложение Б к заданию ЛР 2). Он работает и локально, в Jupyter или VS Code.

Что делает ячейка ниже:
1. В Colab ставит недостающие библиотеки и подключает Google Диск. Colab попросит разрешение на доступ к Диску: выберите тот же аккаунт Google, под которым открыт Colab.
2. Находит рабочую папку группы на Диске. Все данные и результаты сохраняются туда, поэтому после отключения сессии ничего не пропадёт.
3. Создаёт внутри неё ту же структуру папок, что и в репозитории: `data/raw`, `data/manual`, `data/processed`, `docs/img`, `docs/maps`.

**Перед первым запуском** впишите в `PROJECT_FOLDER` имя папки группы на Google Диске.

Запуск всего ноутбука: **Среда выполнения → Выполнить все** (Runtime → Run all).

In [ ]:
# === Настройка среды: запустите эту ячейку первой ===
import sys, subprocess
from pathlib import Path

# Имя папки группы в «Моём диске» Google. Обычно совпадает с именем репозитория.
PROJECT_FOLDER = "pm-lab-00-aquamap"

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", 'geopandas>=1.0', 'mapclassify', 'folium'], check=True)
    from google.colab import drive
    drive.mount("/content/drive")
    BASE = Path("/content/drive/MyDrive") / PROJECT_FOLDER
    if not BASE.exists():
        raise FileNotFoundError(
            f"На Google Диске нет папки «{PROJECT_FOLDER}». Создайте её или добавьте ярлык "
            "общей папки группы в «Мой диск» (приложение Б, шаг 2) и запустите ячейку снова.")
else:
    BASE = Path("..")   # локальный запуск из папки notebooks/ репозитория

RAW, MANUAL, PROCESSED = BASE / "data" / "raw", BASE / "data" / "manual", BASE / "data" / "processed"
IMG, MAPS = BASE / "docs" / "img", BASE / "docs" / "maps"
for folder in (RAW, MANUAL, PROCESSED, IMG, MAPS):
    folder.mkdir(parents=True, exist_ok=True)
print("Среда:", "Google Colab" if IN_COLAB else "локальная", "· рабочая папка:", BASE.resolve())

In [ ]:
import geopandas as gpd
import pandas as pd
import matplotlib.pyplot as plt

WALK_DISTANCE_M = 500          # пешая доступность
PARK_DISTANCE_M = 100          # точка считается «в парке», если ближе этого расстояния к нему

boundary  = gpd.read_file(PROCESSED / "boundary.gpkg")
METRIC_CRS = boundary.estimate_utm_crs()

districts = gpd.read_file(PROCESSED / "districts.gpkg").to_crs(METRIC_CRS)[["name", "geometry"]]
points    = gpd.read_file(PROCESSED / "water_points.gpkg").to_crs(METRIC_CRS)
parks     = gpd.read_file(PROCESSED / "parks.gpkg").to_crs(METRIC_CRS)
homes     = gpd.read_file(PROCESSED / "residential.gpkg").to_crs(METRIC_CRS)
homes["geometry"] = homes.geometry.representative_point()   # здание как точка
print(len(districts), "районов,", len(points), "точек,", len(parks), "парков,", len(homes), "жилых зданий")

## Вопрос 1. Точки по районам и плотность

In [ ]:
districts["area_km2"] = districts.area / 1e6
zones = districts[["name", "geometry"]].rename(columns={"name": "district"})   # слой районов для объединений
pts_in = gpd.sjoin(points, zones, predicate="within", how="inner")
districts["points"] = districts["name"].map(pts_in.groupby("district").size()).fillna(0).astype(int)
districts["points_per_10km2"] = (districts["points"] / districts["area_km2"] * 10).round(2)

### Дополнительно: точки на 10 тыс. жителей

Если в `data/manual/population_by_district.csv` есть численность населения по районам (столбцы `district`, `population`, `year`, `source`), ноутбук посчитает точки на 10 тыс. жителей. Названия районов в таблице должны совпадать с названиями в слое районов.

In [ ]:
POP_FILE = MANUAL / "population_by_district.csv"
pop = pd.read_csv(POP_FILE) if POP_FILE.exists() else pd.DataFrame()
if len(pop):
    districts = districts.merge(pop[["district", "population"]], left_on="name", right_on="district",
                                how="left").drop(columns="district")
    districts["points_per_10k"] = (districts["points"] / districts["population"] * 10_000).round(2)
    missing = districts.loc[districts["population"].isna(), "name"].tolist()
    if missing:
        print("Нет численности для районов:", missing, "— проверьте названия в CSV")
else:
    print("Таблица населения не заполнена, расчёт на 10 тыс. жителей пропущен")
districts.drop(columns="geometry")

## Вопрос 2. Жилые здания в пешей доступности

In [ ]:
nearest = gpd.sjoin_nearest(homes, points[["geometry"]], how="left", distance_col="dist_m")
nearest = nearest[~nearest.index.duplicated()]          # при равных расстояниях оставляем одно совпадение
homes["dist_m"] = nearest["dist_m"]
homes["covered"] = homes["dist_m"] <= WALK_DISTANCE_M

h = gpd.sjoin(homes, zones, predicate="within", how="inner")
cov = h.groupby("district").agg(homes=("covered", "size"), covered=("covered", "sum"),
                            median_dist_m=("dist_m", "median"))
cov["covered_pct"] = (cov["covered"] / cov["homes"] * 100).round(1)
districts = districts.merge(cov, left_on="name", right_index=True, how="left")

print(f"По городу: {homes['covered'].mean() * 100:.1f}% жилых зданий ближе {WALK_DISTANCE_M} м к точке")
districts.drop(columns="geometry").sort_values("covered_pct").round(1)

## Вопрос 3. Парки без питьевой воды

In [ ]:
parks = parks[parks.area > 10_000].copy()               # парки больше 1 га
parks["area_ha"] = (parks.area / 1e4).round(1)
parks["has_water"] = parks.geometry.buffer(PARK_DISTANCE_M).intersects(points.union_all())
no_water = parks[~parks["has_water"]].sort_values("area_ha", ascending=False)
print(f"Парков больше 1 га: {len(parks)}, без питьевой воды: {len(no_water)}")
no_water[["name", "area_ha"]].head(10)

## Карты

In [ ]:
fig, ax = plt.subplots(figsize=(10, 10))
districts.plot(ax=ax, column="covered_pct", cmap="RdYlGn", legend=True, edgecolor="white",
               legend_kwds={"label": f"% жилых зданий ближе {WALK_DISTANCE_M} м к питьевой воде", "shrink": 0.6})
no_water.plot(ax=ax, color="none", edgecolor="darkred", linewidth=1)
points.plot(ax=ax, color="navy", markersize=5)
for _, r in districts.iterrows():
    p = r.geometry.representative_point()
    ax.annotate(r["name"].replace(" район", ""), (p.x, p.y), ha="center", fontsize=8)
ax.set_title("Доступность питьевой воды по районам (© участники OpenStreetMap)")
ax.set_axis_off()
fig.savefig(IMG / "coverage_by_district.png", dpi=150, bbox_inches="tight")

In [ ]:
m = districts.to_crs(4326).explore(column="covered_pct", cmap="RdYlGn", legend=True,
                                   tooltip=["name", "points", "points_per_10km2", "covered_pct"],
                                   name="Районы")
no_water.to_crs(4326).explore(m=m, color="darkred", name="Парки без воды", tooltip=["name"])
points.to_crs(4326).explore(m=m, color="navy", name="Точки воды", tooltip=["name", "amenity"])
import folium; folium.LayerControl().add_to(m)
m.save(MAPS / "analysis_v0.html")
m

## Выводы (заполните)

1. Короткий ответ на каждый вопрос (с цифрами).
2. Насколько можно доверять результату: какие ограничения данных влияют на выводы.
3. Что это значит для продукта: какие истории бэклога подтвердились, какие нужно добавить или убрать.
4. Какие риски из реестра изменились.

Карта `docs/img/coverage_by_district.png` вставляется в отчёт и в презентацию для заказчика.

## Перенос результатов в репозиторий

Карты из `docs/img` и `docs/maps` должны попасть в репозиторий через Pull Request. В Colab ячейка ниже соберёт их в архив `results_03.zip` и скачает на компьютер. Распакуйте архив и загрузите файлы в те же папки репозитория в своей ветке: **Add file → Upload files** (подробно в приложении Б, шаг 6).

Файлы больше 10 МБ в репозиторий не загружайте: ячейка о них предупредит. При локальной работе результаты уже лежат в папках репозитория, ячейку можно пропустить.

In [ ]:
import shutil
FOLDERS = [IMG, MAPS]
if IN_COLAB:
    from google.colab import files
    pack = Path("/content/results_03")
    shutil.rmtree(pack, ignore_errors=True)
    for folder in FOLDERS:
        for f in folder.rglob("*"):
            if f.is_file() and f.name != ".gitkeep":
                if f.stat().st_size > 10 * 1024 * 1024:
                    print(f"Пропущен (больше 10 МБ): {f.relative_to(BASE)}")
                    continue
                target = pack / f.relative_to(BASE)
                target.parent.mkdir(parents=True, exist_ok=True)
                shutil.copy2(f, target)
    archive = shutil.make_archive(str(pack), "zip", pack)
    print("В архиве:", sorted(str(p.relative_to(pack)) for p in pack.rglob("*") if p.is_file()))
    files.download(archive)
else:
    print("Локальный запуск: результаты уже лежат в папках репозитория")